<a href="https://colab.research.google.com/github/yashnaravade/Python/blob/master/Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

In [4]:
# 1. Purge the incompatible Python 3.12 apt and pip binaries
!apt-get remove python3-libtorrent --yes
!pip uninstall lbry-libtorrent -y

# 2. Install a clean, standard distribution wheel
!pip install libtorrent

# 3. Mount your Google Drive storage target
from google.colab import drive
drive.mount('/content/drive')

# 4. Verify initialization
import libtorrent as lt
ses = lt.session()
ses.listen_on(6881, 6891)
print(f"Success! Libtorrent {lt.__version__} initialized correctly.")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libboost-python1.74.0 libtorrent-rasterbar2.0
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  python3-libtorrent
0 upgraded, 0 newly installed, 1 to remove and 1 not upgraded.
After this operation, 3,437 kB disk space will be freed.
(Reading database ... 118233 files and directories currently installed.)
Removing python3-libtorrent (2.0.5-5) ...
Found existing installation: lbry-libtorrent 1.2.4
Uninstalling lbry-libtorrent-1.2.4:
  Successfully uninstalled lbry-libtorrent-1.2.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 22.7 MB/s  0:00:00
Mounted at /content/drive
Success! Libtorrent 2.0.11.0 initialized correctly.


/tmp/ipykernel_2026/438622847.py:15: DeprecationWarning: listen_on() is deprecated
  ses.listen_on(6881, 6891)


# 1. Purge the incompatible Python 3.12 apt and pip binaries
!apt-get remove python3-libtorrent --yes
!pip uninstall lbry-libtorrent -y

# 2. Install a clean, standard distribution wheel
!pip install libtorrent

# 3. Mount your Google Drive storage target
from google.colab import drive
drive.mount('/content/drive')

# 4. Verify initialization
import libtorrent as lt
ses = lt.session()
ses.listen_on(6881, 6891)
print(f"Success! Libtorrent {lt.__version__} initialized correctly.")
### Install libtorrent and Initialize Session

### Mount Google Drive
To stream files we need to mount Google Drive.

In [5]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Add From Torrent File
You can run this cell to add more files as many times as you want

In [7]:
from google.colab import files

# 1. Initialize the missing downloads list
if 'downloads' not in locals() and 'downloads' not in globals():
    downloads = []

source = files.upload()

# 2. Add the torrent to the session and append to the list
params = {
    "save_path": "/content/drive/My Drive/Torrent",
    "ti": lt.torrent_info(list(source.keys())[0]),
}
downloads.append(ses.add_torrent(params))
print("Torrent added to download queue successfully!")


Saving Daily Dose of Sunshine (2023) - Korean with English subs, 1080p.torrent to Daily Dose of Sunshine (2023) - Korean with English subs, 1080p (1).torrent
Torrent added to download queue successfully!


### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [ ]:
params = {"save_path": "/content/drive/My Drive/Torrent"}

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    downloads.append(
        lt.add_magnet_uri(ses, magnet_link, params)
    )


### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [ ]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(
        step=0.01, disabled=True, layout=layout, style=style
    )
    for _ in downloads
]
display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        if not download.is_seed():
            s = download.status()

            bar.description = " ".join(
                [
                    download.name(),
                    str(s.download_rate / 1000),
                    "kB/s",
                    state_str[s.state],
                ]
            )
            bar.value = s.progress * 100
        else:
            next_shift -= 1
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close() # Seems to be not working in Colab (see https://github.com/googlecolab/colabtools/issues/726#issue-486731758)
            download_bars.remove(bar)
            print(download.name(), "complete")
    time.sleep(1)


FloatSlider(value=0.0, disabled=True, layout=Layout(width='auto'), step=0.01, style=SliderStyle(description_wi…

/tmp/ipykernel_2026/3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipykernel_2026/3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),
